# Pre-trend diagnostic — PT_violation_het20 (linearity_degree=1)

**Workstream PT · conditional-parallel-trends diagnostic (Reviewer 3.2.3)**

heterogeneous violation, slope +/-0.2 per period by X1, zero on average: the marginal event study is powerless by construction and only the subgroup contrast (PRE_SUBC) can detect it

The estimation model sets `Z = D_it`, which is zero on every pre-treatment row,
so those rows carry **no information about tau** and no post-processing of that
fit can produce a placebo coefficient. This notebook therefore fits the
*unconstrained* specification (`Z = 1[G_i != inf]`, on in every period) as a
**separate diagnostic model**, and reports

$$\\Delta(k) = E\\big[\\tau(X, k) - \\tau(X, -1)\\;\\big|\\;\\text{treated}\\big],
\\qquad k < -1,$$

which is exactly zero under conditional parallel trends. The TWFE event-study
placebo is run on the same replications, for free, as the standard-practice
comparator.

> **Colab:** upload just this notebook and *Run all*.

In [ ]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

In [ ]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.pretrend_runner import run_named
from did_bcf_revision.metrics import compute_metrics

In [ ]:
REPS = 200      # detection rates need replications; 200 gives MCSE <= 0.035
JOBS = 2        # ~1 GB RAM per worker; drop to 1 if the VM is memory-starved

# WITH_ATT also fits the constrained estimation model on the same replications,
# so the ATT bias the violation causes is measured alongside its detection.
# It doubles the MCMC cost -- worth it at degree 1, skip it at 2 and 3.
WITH_ATT = True

# A Colab session is capped at roughly 8 hours. One fit is ~2 min, so a full
# 200-replication WITH_ATT run is ~7 h at JOBS=2 -- close enough to the cap that
# it is worth splitting. Set these to (0, 100) here and (100, 200) in a second
# copy of this notebook; replications are seeded by index, so the two parts
# concatenate into exactly the undivided run and land in separate files.
REP_START, REP_END = 0, 50

summaries = run_named(
    "PT_violation_het20",
    linearity_degree=1,
    reps=REPS,
    jobs=JOBS,
    with_att=WITH_ATT,
    rfx="unit",     # unit intercepts absorb the level gap conditional PT allows
    bcf_params=dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3),
    rep_start=REP_START, rep_end=REP_END,
)
summaries.head()

# Save the completed replication block before displaying the metrics. This is
# the file downloaded automatically when the notebook is run on Colab.
output_file = (
    f"summaries_pretrend_{summaries['setting'].iloc[0]}"
    f"_lin_{int(summaries['linearity_degree'].iloc[0])}"
    f"_reps{REP_START}-{REP_END}.csv"
)
summaries.to_csv(output_file, index=False)
print("wrote", output_file, "| rows:", len(summaries))

try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)

In [ ]:
# `reject05` on the PRE rows is the diagnostic's **size** when the true
# differential slope is 0 and its **detection rate** otherwise; `any_bonf` is the
# per-replication decision rule (any pre-period significant, Bonferroni-scaled).
metrics = compute_metrics(summaries)
pre = metrics[metrics.estimand_type == "PRE"]
pre[["method", "estimand_id", "mean_true", "bias", "cover95",
     "reject05", "mcse_reject05", "role"]].sort_values(["estimand_id", "method"])

## Conditional check

`Delta(k)` within covariate subgroups. A marginal event study cannot produce
this without pre-specifying the interactions; here it comes out of the same fit,
and it is the version that matches what the estimator actually assumes.

In [ ]:
sub = metrics[metrics.estimand_type == "PRE_SUB"]
sub[["estimand_id", "mean_true", "bias", "emp_sd", "cover95", "reject05"]]

## The conditional *test*

The subgroup levels above are four looks at one fact whenever the violation is
homogeneous: every subgroup moves together, so reporting each one separately
adds nothing. The **contrast** between subgroups is the object that carries a
test. It is exactly zero under any violation that is constant in `X` — including
the whole `PT_violation_g*` grid — so its rejection rate there and under
`PT_hold` is a *size*, and its rejection rate under `PT_violation_het*` is the
power of something no marginal event study can compute at all.

Read this table against the `PRE` table above. Under `PT_violation_het*` the
aggregate rows (both `pretrend` and `twfe_es`) should be at their size while
`X1_any` climbs: the violation is engineered to cancel in the average, so an
event study is powerless there by construction, not by bad luck.

In [ ]:
subc = metrics[metrics.estimand_type == "PRE_SUBC"]
subc[["estimand_id", "mean_true", "bias", "emp_sd", "cover95",
      "reject05", "mcse_reject05"]].sort_values("estimand_id")